# Chapter 7: Attention Overview

This notebook shows the chapter 7 decision chain: baseline attention, blockwise attention, and a simple comparison against the reference implementation.
The point is to make the cost structure visible before discussing FlashAttention and fusion.

## 1. Import the shared helpers

We reuse the chapter 7 helpers so this notebook stays aligned with the scripts and the chapter text.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "HANDOFF.md").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.ch07_attention_core import attention_blockwise, attention_reference, benchmark, make_attention_inputs
from examples.benchmark_record_template import set_benchmark_record

batch = 1
seq_len = 32
dim = 64
q, k, v = make_attention_inputs(batch=batch, seq_len=seq_len, dim=dim, seed=0)
print(f"batch={batch}, seq_len={seq_len}, dim={dim}")

## 2. Run the reference and blockwise versions

The goal here is not to tune anything yet. It is to see how the baseline and blockwise versions compare, and how much structure the blockwise version already exposes.

In [ ]:
ref_out, ref_ms = benchmark(attention_reference, q, k, v, runs=3)
blk_out, blk_ms = benchmark(attention_blockwise, q, k, v, runs=3)

diff = abs(ref_out - blk_out).max()
print(f"Reference: {ref_ms:.3f} ms")
print(f"Blockwise:  {blk_ms:.3f} ms")
print(f"Max diff:   {diff:.6f}")

## 3. Interpret the result

If the blockwise result is numerically close to the reference, then the next question is not "is it correct?" but "where is the time going?".
That is the chapter 7 transition: once correctness is stable, the remaining problem is data flow, intermediate state, and fusion boundaries.

In [ ]:
record = set_benchmark_record(
    scenario="chapter 7 attention overview",
    operator="attention",
    platform="NVIDIA",
    target="cuda",
    dtype="float32",
    shape=f"batch={batch}, seq={seq_len}, dim={dim}",
    baseline="reference attention",
    optimization="blockwise attention",
)

for key in ["scenario", "operator", "platform", "target", "dtype", "shape", "baseline", "optimization"]:
    print(f"{key}: {record[key]}")

## 4. What this notebook is for

This notebook is intentionally light. It is here to show the chapter's first comparison point and to prepare the reader for FlashAttention, fusion, and the optimization order discussed in the text.